# EMDGrid Colab Demo: EMD-L1 and Knothe-Rosenblatt Transport Plans

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tvercaut/emdgrid/blob/main/example/colab_demo.ipynb)

This notebook demonstrates how to compile and run **emdgrid** in Google Colab to compute exact EMD-L1 and Knothe-Rosenblatt (KR) optimal transport plans on 2D grid histograms, visualize their sparsity patterns, plot dense transport matrices with marginal distributions, and show barycentric projections of mass transport.

In [ ]:
# Install build dependencies and compile emdgrid in Google Colab if needed
import os
import sys

if 'google.colab' in sys.modules:
    !apt-get update -qq && apt-get install -y -qq cmake build-essential
    !git clone https://github.com/tvercaut/emdgrid.git
    %cd emdgrid
    !cmake -B build -DEMDGRID_BUILD_PYTHON_BINDINGS=ON -DBUILD_TESTING=OFF -DEMDGRID_BUILD_EXAMPLES=OFF
    !cmake --build build -j
    sys.path.insert(0, os.path.abspath('build/bindings/python'))
else:
    # Add local build path if running locally
    possible_paths = [
        os.path.abspath('build/bindings/python'),
        os.path.abspath('../build/bindings/python'),
    ]
    for p in possible_paths:
        if os.path.exists(p) and p not in sys.path:
            sys.path.insert(0, p)

import pyemdgrid

print(f"Successfully loaded pyemdgrid version: {pyemdgrid.version()}")

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import scipy.sparse

# Set random seed for reproducibility
np.random.seed(42)

## 1. Create 10x10 Grid Histograms

We generate two 10x10 normalized grid histograms ($H_1$ and $H_2$) representing probability distributions on a 2D integer grid.

In [ ]:
# Create a 10x10 spatial grid
grid_size = 10
x, y = np.meshgrid(np.arange(grid_size), np.arange(grid_size), indexing='ij')

# Define two 2D Gaussian distributions on the 10x10 grid
mu1, sigma1 = (2.5, 2.5), 1.5
mu2, sigma2 = (6.5, 6.5), 1.5

h1 = np.exp(-((x - mu1[0]) ** 2 + (y - mu1[1]) ** 2) / (2 * sigma1 ** 2))
h2 = np.exp(-((x - mu2[0]) ** 2 + (y - mu2[1]) ** 2) / (2 * sigma2 ** 2))

# Normalize histograms so total mass is 1.0
h1 /= h1.sum()
h2 /= h2.sum()

# Plot the source and target 10x10 histograms
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

im1 = axes[0].imshow(h1, cmap='viridis', origin='lower')
axes[0].set_title('Source Histogram $H_1$ (10x10)')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
fig.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(h2, cmap='viridis', origin='lower')
axes[1].set_title('Target Histogram $H_2$ (10x10)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
fig.colorbar(im2, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 2. Compute Transport Plans with EMD-L1 and Knothe-Rosenblatt

We compute sparse transport plans for both solvers:
1. **EMD-L1**: Exact tree-based network simplex optimal transport solver under $L_1$ ground distance.
2. **Knothe-Rosenblatt (KR)**: Fast $N$-D heuristic optimal transport solver.

In [ ]:
# Compute EMD-L1 transport plan and cost
cost_emd, plan_emd = pyemdgrid.emd_l1(h1, h2, return_transport_plan=True)

# Compute Knothe-Rosenblatt transport plan and cost
cost_kr, plan_kr = pyemdgrid.knothe_rosenblatt(h1, h2, metric='l1', return_transport_plan=True)

print(f"EMD-L1 Cost : {cost_emd:.6f} | Non-zero entries (NNZ): {plan_emd.nnz}")
print(f"KR Cost     : {cost_kr:.6f} | Non-zero entries (NNZ): {plan_kr.nnz}")

## 3. Visualize Sparsity Patterns

The transport plans are returned as sparse matrices of shape (100, 100) representing the flow from the 100 flattened source cells to the 100 flattened target cells.
We plot their sparsity patterns using `plt.spy()`. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].spy(plan_emd, markersize=3, color='navy')
axes[0].set_title(f"EMD-L1 Transport Plan Sparsity Pattern\nCost: {cost_emd:.4f} | NNZ: {plan_emd.nnz}")
axes[0].set_xlabel("Target Cell Index (0..99)")
axes[0].set_ylabel("Source Cell Index (0..99)")

axes[1].spy(plan_kr, markersize=3, color='darkred')
axes[1].set_title(f"Knothe-Rosenblatt Transport Plan Sparsity Pattern\nCost: {cost_kr:.4f} | NNZ: {plan_kr.nnz}")
axes[1].set_xlabel("Target Cell Index (0..99)")
axes[1].set_ylabel("Source Cell Index (0..99)")

plt.tight_layout()
plt.show()

## 4. Dense Transport Matrices with Marginal Distributions

We display the dense transport matrices $P_{ij}$ with a shared colormap range $[0, v_{\max}]$ where $v_{\max} = \max(P_{\mathrm{EMD}}, P_{\mathrm{KR}})$ using the `viridis` colormap.
Around each matrix, we plot the marginal distributions:
- **Top**: Target marginal $\sum_i P_{ij}$ (columns) compared to target distribution $H_2$.
- **Right**: Source marginal $\sum_j P_{ij}$ (rows) compared to source distribution $H_1$.

In [ ]:
emd_dense = plan_emd.toarray()
kr_dense = plan_kr.toarray()

# Shared color scale maximum across both plans
vmax = max(emd_dense.max(), kr_dense.max())

def plot_dense_plan_with_marginals(plan_dense, title, cmap, cost, nnz):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    im = ax.imshow(plan_dense, cmap=cmap, vmin=0, vmax=vmax, origin='upper', aspect='equal')
    ax.set_xlabel("Target Cell Index (0..99)")
    ax.set_ylabel("Source Cell Index (0..99)")
    ax.set_title(f"{title}\nCost: {cost:.4f} | NNZ: {nnz}", pad=20)

    divider = make_axes_locatable(ax)

    # Top histogram: Target marginal vs H2
    ax_top = divider.append_axes("top", 1.0, pad=0.1, sharex=ax)
    target_marg = plan_dense.sum(axis=0)
    ax_top.plot(np.arange(100), target_marg, color='blue', lw=1.5, label='Plan Marginal $\\sum_i P_{ij}$')
    ax_top.plot(np.arange(100), h2.ravel(), color='black', linestyle='--', lw=1.2, label='Target $H_2$')
    ax_top.set_ylabel("Mass")
    ax_top.legend(fontsize=8, loc='upper right')
    ax_top.xaxis.set_visible(False)

    # Right histogram: Source marginal vs H1
    ax_right = divider.append_axes("right", 1.0, pad=0.1, sharey=ax)
    source_marg = plan_dense.sum(axis=1)
    ax_right.plot(source_marg, np.arange(100), color='blue', lw=1.5, label='Plan Marginal $\\sum_j P_{ij}$')
    ax_right.plot(h1.ravel(), np.arange(100), color='black', linestyle='--', lw=1.2, label='Source $H_1$')
    ax_right.set_xlabel("Mass")
    ax_right.invert_yaxis()
    ax_right.legend(fontsize=8, loc='upper right')
    ax_right.yaxis.set_visible(False)

    # Colorbar
    cax = divider.append_axes("right", 0.15, pad=0.8)
    fig.colorbar(im, cax=cax)

    plt.show()

plot_dense_plan_with_marginals(emd_dense, "EMD-L1 Transport Matrix", "viridis", cost_emd, plan_emd.nnz)
plot_dense_plan_with_marginals(kr_dense, "Knothe-Rosenblatt Transport Matrix", "viridis", cost_kr, plan_kr.nnz)

## 5. Mass Transport Visualization & Barycentric Map Projection

To visualize how mass is transported spatially from $H_1$ to $H_2$:
1. **Barycentric Map Scatter Plot**: Each source cell $i=(x_i, y_i)$ is displayed as a point at its original location, and as a point at its barycentric projected target position $T(x_i, y_i) = \sum_j P_{ij} (x_j, y_j) / h_{1,i}$, with point sizes scaled by source mass $h_{1,i}$.
2. **Mass-Weighted Quiver Plot**: Displacement vectors connecting source cells to their barycentric target locations, with arrow opacity scaled by source cell mass $h_1(x,y)$.

In [ ]:
x_flat = x.ravel().astype(float)
y_flat = y.ravel().astype(float)
h1_flat = h1.ravel()

mask = h1_flat > 1e-12

def compute_barycentric_map(plan_dense):
    target_x = np.zeros_like(h1_flat)
    target_y = np.zeros_like(h1_flat)
    target_x[mask] = (plan_dense[mask] @ x_flat) / h1_flat[mask]
    target_y[mask] = (plan_dense[mask] @ y_flat) / h1_flat[mask]
    return target_x, target_y

tx_emd, ty_emd = compute_barycentric_map(emd_dense)
tx_kr, ty_kr = compute_barycentric_map(kr_dense)

# Figure 1: Barycentric Projection Scatter Map
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, tx, ty, title in zip(axes, [tx_emd, tx_kr], [ty_emd, ty_kr], ["EMD-L1 Barycentric Map", "Knothe-Rosenblatt Barycentric Map"]):
    # Plot target distribution contour in background
    ax.contourf(x, y, h2, cmap='Greys', alpha=0.3)

    # Plot source points (blue) and mapped target points (red)
    s_sizes = 5000 * h1_flat[mask]
    ax.scatter(x_flat[mask], y_flat[mask], s=s_sizes, color='blue', alpha=0.6, label='Source $H_1$')
    ax.scatter(tx[mask], ty[mask], s=s_sizes, color='red', alpha=0.6, label='Barycentric Target $T(H_1)$')

    # Draw lines connecting source to barycentric target
    for i in np.where(mask)[0]:
        ax.plot([x_flat[i], tx[i]], [y_flat[i], ty[i]], color='black', alpha=min(1.0, h1_flat[i] * 50), lw=1)

    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.legend()

plt.tight_layout()
plt.show()

# Figure 2: Mass-Weighted Displacement Quiver Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, tx, ty, title, cmap in zip(axes, [tx_emd, tx_kr], [ty_emd, ty_kr],
                                   ["EMD-L1 Displacement Field", "Knothe-Rosenblatt Displacement Field"],
                                   ["Blues", "Reds"]):
    ax.imshow(h1.T, cmap=cmap, origin='lower', alpha=0.5)

    dx = tx - x_flat
    dy = ty - y_flat

    # Normalize alpha/opacity by cell mass
    max_mass = h1_flat.max()
    arrow_color = 'navy' if 'Blues' in cmap else 'darkred'
    for i in np.where(mask)[0]:
        alpha_val = min(1.0, max(0.1, h1_flat[i] / max_mass))
        ax.quiver(x_flat[i], y_flat[i], dx[i], dy[i],
                   angles='xy', scale_units='xy', scale=1,
                   color=arrow_color, alpha=alpha_val, width=0.01)

    ax.set_title(f"{title}\n(Arrow Opacity Scaled by Source Mass)")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()